# 04 — Bayesian model (pipeline)

**Model:** BYM2 Gaussian regression on `log1p` of LODES jobs reachable at the **primary** travel-time threshold (`accessibility.travel_time_threshold_min`, default **45** min). Random effects follow the [PyMC NYC BYM](https://www.pymc.io/projects/examples/en/latest/spatial/nyc_bym.html) parameterization: `σ · (√(1−ρ)·θ + √(ρ/s)·φ)` with `φ ~ ICAR(W)` and **Riebler scaling** `s` (not the unscaled `√ρ·φ` form).

**Reads:** latest `data/processed/accessibility/tract_accessibility_bundle__*.parquet`, latest `artifacts/tables/eda__acs_sd_tract_attributes__*.csv`, TIGER tracts (SD county), `configs/defaults.yaml` + `san_diego.yaml`.

**Writes:** `data/processed/posteriors/<run_id>_posterior_{samples,summary}.parquet`, `<run_id>_idata.nc`, `artifacts/tables/pipeline__04_model_diagnostics__<run_id>.csv`, trace + maps under `artifacts/figures/`.

**Previous:** [`03_accessibility_computation.ipynb`](03_accessibility_computation.ipynb) · **Next:** [`05_posterior_analysis.ipynb`](05_posterior_analysis.ipynb)

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

REPO_ROOT = next(
    (d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "configs" / "san_diego.yaml").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find configs/san_diego.yaml.")
sys.path.insert(0, str(REPO_ROOT))

from src.utils.config import artifact_run_id, load_merged_config  # noqa: E402

CONFIG = load_merged_config(REPO_ROOT)
MODEL_CFG = CONFIG.get("model", {})
ACC_CFG = CONFIG.get("accessibility", {})
T_PRIMARY = int(ACC_CFG.get("travel_time_threshold_min", 45))
RID = artifact_run_id()
FAST = os.environ.get("PIPELINE_FAST_MCMC", "").strip() in ("1", "true", "yes")
if FAST:
    MODEL_CFG = {**MODEL_CFG, "draws": 80, "tune": 80, "chains": 2}

POSTERIOR_DIR = REPO_ROOT / "data" / "processed" / "posteriors"
POSTERIOR_DIR.mkdir(parents=True, exist_ok=True)
ART_TAB = REPO_ROOT / "artifacts" / "tables"
ART_FIG = REPO_ROOT / "artifacts" / "figures"
ART_TAB.mkdir(parents=True, exist_ok=True)
ART_FIG.mkdir(parents=True, exist_ok=True)

SEED = int(MODEL_CFG.get("random_seed", 42))
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

print(f"REPO_ROOT={REPO_ROOT}")
print(f"RUN_ID={RID}  T_PRIMARY={T_PRIMARY}min  FAST_MCMC={FAST}")
print(f"MCMC draws={MODEL_CFG.get('draws')} tune={MODEL_CFG.get('tune')} chains={MODEL_CFG.get('chains')}")


In [ ]:
# Load accessibility bundle (latest by filename date), ACS attributes, TIGER tracts
import geopandas as gpd
from IPython.display import display

census_cfg = CONFIG.get("census", {})
state_fips = str(census_cfg.get("state_fips", CONFIG.get("state_fips", "06"))).zfill(2)
county_fips = str(census_cfg.get("county_fips", CONFIG.get("county_fips", "073"))).zfill(3)
acs_year = int(census_cfg.get("acs_year", 2023))

bundles = sorted((REPO_ROOT / "data" / "processed" / "accessibility").glob("tract_accessibility_bundle__*.parquet"))
if not bundles:
    raise FileNotFoundError("No tract_accessibility_bundle__*.parquet — run notebook 03 first.")
bundle_path = bundles[-1]

acs_files = sorted(ART_TAB.glob("eda__acs_sd_tract_attributes__*.csv"))
if not acs_files:
    raise FileNotFoundError("No eda__acs_sd_tract_attributes__*.csv — run EDA notebook 03 first.")
acs_path = acs_files[-1]

tiger_dir = REPO_ROOT / "data" / "raw" / "census" / f"tl_{acs_year}_{state_fips}_tract"
tiger_shp = tiger_dir / f"tl_{acs_year}_{state_fips}_tract.shp"
if not tiger_shp.is_file():
    raise FileNotFoundError(f"Missing TIGER shapefile: {tiger_shp}")

acc = pd.read_parquet(bundle_path)
acc["GEOID"] = acc["GEOID"].astype(str).str.zfill(11)

acs = pd.read_csv(acs_path)
acs["GEOID"] = acs["GEOID"].astype(str).str.zfill(11)

cov_raw = [
    "poverty_rate",
    "no_vehicle_hh_rate",
    "pct_hispanic",
    "pct_black",
    "pct_nh_white",
    "B19013_001E",
    "B01003_001E",
    "disadvantage_z",
]
usecols = ["GEOID"] + [c for c in cov_raw if c in acs.columns]
acs = acs[usecols]

tracts_all = gpd.read_file(tiger_shp)
tracts_all["GEOID"] = tracts_all["GEOID"].astype(str).str.zfill(11)
tracts_sd = tracts_all[tracts_all["COUNTYFP"] == county_fips].copy()

df = acc.merge(acs, on="GEOID", how="inner", suffixes=("", "_acs"))
df = df.merge(tracts_sd[["GEOID", "geometry"]], on="GEOID", how="inner")

job_cols = {30: "jobs_C000_30min", 45: "jobs_C000_45min", 60: "jobs_C000_60min"}
y_col = job_cols[T_PRIMARY]
if y_col not in df.columns:
    raise KeyError(f"Bundle missing column {y_col!r}")

# Population density (people / km²) in projected CRS
tg = gpd.GeoDataFrame(df[["GEOID", "geometry"]], geometry="geometry", crs="EPSG:4326")
tg = tg.to_crs(3310)
area_km2 = tg.geometry.area / 1e6
pop = pd.to_numeric(df["B01003_001E"], errors="coerce").clip(lower=1.0)
df["pop_density_km2"] = pop / area_km2.replace(0, np.nan)

# Log median income (impute missing)
inc = pd.to_numeric(df["B19013_001E"], errors="coerce")
inc = inc.fillna(np.nanmedian(inc))
df["log_median_income"] = np.log(inc.clip(lower=1.0))

# Design matrix: z-scored covariates for the model
cov_model = [
    "poverty_rate",
    "no_vehicle_hh_rate",
    "pct_hispanic",
    "pct_black",
    "pct_nh_white",
    "log_median_income",
    "log_pop_density",
]
df["log_pop_density"] = np.log1p(df["pop_density_km2"].fillna(0.0))

for c in cov_model:
    if c not in df.columns:
        raise KeyError(c)

X_raw = df[cov_model].apply(pd.to_numeric, errors="coerce").astype(float)
X_mean = X_raw.mean()
X_std = X_raw.std(ddof=1).replace(0, 1.0)
X_z = (X_raw - X_mean) / X_std
X_z.index = df["GEOID"].astype(str).str.zfill(11)

df["y_raw"] = pd.to_numeric(df[y_col], errors="coerce").fillna(0.0)
df["y_log"] = np.log1p(df["y_raw"])

# Quartile thresholds on observed job counts (city-specific "desert" cut)
q25 = {t: float(np.percentile(pd.to_numeric(df[job_cols[t]], errors="coerce").fillna(0.0), 25)) for t in (30, 45, 60)}

n_acc = len(acc)
n_merge = len(df)
display(
    pd.DataFrame(
        [
            {"stage": "accessibility rows", "n": n_acc},
            {"stage": "inner join acc + acs + tiger", "n": n_merge},
        ]
    )
)

miss_acs = set(acc["GEOID"]) - set(acs["GEOID"])
miss_tg = set(acc["GEOID"]) - set(tracts_sd["GEOID"])
if miss_acs or miss_tg:
    print("GEOs in accessibility but not ACS:", len(miss_acs), "not in TIGER:", len(miss_tg))

if n_merge < 700:
    raise RuntimeError(f"Only {n_merge} tracts merged — expected ≥700; check upstream joins.")

print("Bundle:", bundle_path.relative_to(REPO_ROOT))
print("ACS:", acs_path.relative_to(REPO_ROOT))
print("Response:", y_col, "| q25 jobs (30/45/60 min):", q25)

In [ ]:
# Queen adjacency + Riebler scaling factor (connected graph after island fix)
import matplotlib.pyplot as plt

from src.modeling.spatial import adjacency_from_queen, scaling_factor_sp

tracts_gdf = gpd.GeoDataFrame(df[["GEOID", "geometry"]], geometry="geometry", crs="EPSG:4326")
W, geoids_sp, diag_sp, tracts_ordered = adjacency_from_queen(tracts_gdf, id_col="GEOID", connect_islands=True)

scaling_factor = scaling_factor_sp(W)
print({k: v for k, v in diag_sp.items() if k != "geoid_order"})
print("Riebler scaling_factor:", round(scaling_factor, 6))

df_ix = df.set_index("GEOID").loc[geoids_sp].reset_index()
y_log = df_ix["y_log"].to_numpy(dtype=np.float64)
X = X_z.loc[df_ix["GEOID"].astype(str).str.zfill(11)].to_numpy(dtype=np.float64)
cov_names = list(X_z.columns)

deg = W.sum(axis=1)
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(deg, bins=20, color="steelblue", edgecolor="white")
ax.set_xlabel("Queen degree (# neighbors)")
ax.set_title("Tract contiguity degree distribution")
plt.tight_layout()
deg_path = ART_FIG / f"pipeline__04_degree_hist__{RID}.png"
fig.savefig(deg_path, dpi=150)
plt.close(fig)
print("Saved", deg_path.relative_to(REPO_ROOT))


In [ ]:
# PyMC BYM2 + Gaussian likelihood on log1p jobs (primary threshold)
from IPython.display import display

import pymc as pm

from src.modeling.tract_bym import build_tract_bym_normal

model = build_tract_bym_normal(
    W=W,
    scaling_factor=scaling_factor,
    X=X,
    y_log=y_log,
    geoids=geoids_sp,
    cov_names=cov_names,
)

print("Covariate order (z-scored in model):", cov_names)

try:
    gv = pm.model_to_graphviz(model)
    display(gv)
except Exception as e:
    print("model_to_graphviz skipped:", e)


In [ ]:
# MCMC — save NetCDF immediately for crash safety
idata_path = POSTERIOR_DIR / f"{RID}_idata.nc"

with model:
    idata = pm.sample(
        draws=int(MODEL_CFG["draws"]),
        tune=int(MODEL_CFG["tune"]),
        chains=int(MODEL_CFG["chains"]),
        target_accept=float(MODEL_CFG["target_accept"]),
        random_seed=SEED,
        return_inferencedata=True,
        progressbar=True,
    )

idata.to_netcdf(idata_path)
print("Saved", idata_path.relative_to(REPO_ROOT))


In [ ]:
# Convergence diagnostics (ICAR φ per-tract R-hat is often noisy — prioritize globals)
import arviz as az
import matplotlib.pyplot as plt
from IPython.display import display

div = int(idata.sample_stats["diverging"].sum().values.item())
print("Divergences:", div)

try:
    print("BFMI by chain:", az.bfmi(idata))
except Exception as e:
    print("BFMI:", e)

sum_tab = az.summary(
    idata,
    var_names=["alpha", "beta", "rho", "sigma", "sigma_obs"],
    round_to=4,
)
display(sum_tab)

out_diag = sum_tab.reset_index()
c0 = out_diag.columns[0]
out_diag = out_diag.rename(columns={c0: "parameter"})
out_diag["divergences_total"] = div
out_diag.to_csv(ART_TAB / f"pipeline__04_model_diagnostics__{RID}.csv", index=False)
print("Wrote", (ART_TAB / f"pipeline__04_model_diagnostics__{RID}.csv").relative_to(REPO_ROOT))

az.plot_trace(idata, var_names=["alpha", "beta", "rho", "sigma", "sigma_obs"], figsize=(12, 10))
trace_path = ART_FIG / f"pipeline__04_trace_plot__{RID}.png"
plt.savefig(trace_path, dpi=150, bbox_inches="tight")
plt.close("all")
print("Wrote", trace_path.relative_to(REPO_ROOT))


In [ ]:
# Posterior summaries + long-format samples (mu on log1p scale = INTERFACES `accessibility`)
mu = idata.posterior["mu"]  # chain, draw, tract
stacked = mu.stack(sample=("chain", "draw"))
mu_np = mu.values
C, D, N = mu_np.shape

thr_log = {t: np.log1p(q25[t]) for t in (30, 45, 60)}
exc = {}
for t in (30, 45, 60):
    thr = thr_log[t]
    exc[t] = (mu_np < thr).mean(axis=(0, 1))

p_desert = exc[T_PRIMARY]

pm_mean = stacked.mean("sample").values
pm_sd = stacked.std("sample").values
qlo = stacked.quantile(0.025, dim="sample").values
qhi = stacked.quantile(0.975, dim="sample").values

summary = pd.DataFrame(
    {
        "GEOID": geoids_sp,
        "posterior_mean_log1p": pm_mean,
        "posterior_sd_log1p": pm_sd,
        "ci_lower_95_log1p": qlo,
        "ci_upper_95_log1p": qhi,
        "posterior_mean_jobs": np.expm1(pm_mean),
        "posterior_sd_jobs_delta": np.exp(pm_mean) * pm_sd,
        "exceedance_prob_30min": exc[30],
        "exceedance_prob_45min": exc[45],
        "exceedance_prob_60min": exc[60],
        "p_transit_desert": p_desert,
    }
)

# Raw demographics for downstream GeoJSON (INTERFACES)
for col in ("B01003_001E", "no_vehicle_hh_rate", "B19013_001E", "poverty_rate"):
    if col in df_ix.columns:
        summary[col] = df_ix[col].values

sum_path = POSTERIOR_DIR / f"{RID}_posterior_summary.parquet"
summary.to_parquet(sum_path, index=False)
print("Wrote", sum_path.relative_to(REPO_ROOT))

chain_idx = np.broadcast_to(np.arange(C)[:, None, None], (C, D, N))
draw_idx = np.broadcast_to(np.arange(D)[None, :, None], (C, D, N))
tract_idx = np.broadcast_to(np.arange(N)[None, None, :], (C, D, N))
geoid_arr = np.array(geoids_sp)
long = pd.DataFrame(
    {
        "chain": chain_idx.ravel(),
        "sample_idx": draw_idx.ravel(),
        "geoid": geoid_arr[tract_idx.ravel()],
        "accessibility": mu_np.ravel(),
    }
)
samp_path = POSTERIOR_DIR / f"{RID}_posterior_samples.parquet"
long.to_parquet(samp_path, index=False)
print("Wrote", samp_path.relative_to(REPO_ROOT), "rows", len(long))


In [ ]:
# Maps + equity Spearman table (posterior vs disadvantage_z)
from scipy import stats

map_gdf = tracts_ordered.copy()
map_gdf["GEOID"] = map_gdf["GEOID"].astype(str).str.zfill(11)
map_gdf = map_gdf.merge(summary[["GEOID", "posterior_mean_jobs", "posterior_sd_log1p", "exceedance_prob_45min"]], on="GEOID")

fig, ax = plt.subplots(figsize=(9, 9))
map_gdf.plot(column="posterior_mean_jobs", cmap="YlOrRd", legend=True, ax=ax, missing_kwds={"color": "lightgrey"})
ax.set_axis_off()
ax.set_title("Posterior mean jobs (expm1 of μ on log1p scale)")
fig_path = ART_FIG / f"pipeline__04_posterior_mean_map__{RID}.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print("Wrote", fig_path.relative_to(REPO_ROOT))

fig, ax = plt.subplots(figsize=(9, 9))
map_gdf.plot(column="posterior_sd_log1p", cmap="viridis", legend=True, ax=ax, missing_kwds={"color": "lightgrey"})
ax.set_axis_off()
ax.set_title("Posterior SD (log1p job scale)")
fig_path2 = ART_FIG / f"pipeline__04_posterior_sd_map__{RID}.png"
fig.savefig(fig_path2, dpi=150, bbox_inches="tight")
plt.close(fig)
print("Wrote", fig_path2.relative_to(REPO_ROOT))

dz = pd.to_numeric(df_ix["disadvantage_z"], errors="coerce")
rows = []
for col in ["posterior_mean_jobs", "posterior_sd_log1p", "exceedance_prob_45min", "p_transit_desert"]:
    pair = pd.DataFrame({"x": summary[col], "z": dz}).dropna()
    if len(pair) > 3:
        r, p = stats.spearmanr(pair["x"], pair["z"])
        rows.append({"variable": col, "spearman_rho": r, "p_value": p})
spear = pd.DataFrame(rows)
display(spear)
spear.to_csv(ART_TAB / f"pipeline__04_equity_spearman__{RID}.csv", index=False)


### Design notes (nb04)

- **BYM2 scaling:** We use `√(ρ/s)·φ` with `s = scaling_factor_sp(W)` from Riebler et al. (2016), matching the official PyMC NYC BYM example — **not** `√ρ·φ` without `s`, which confounds interpretation of `ρ`.
- **One likelihood:** `μ` is fit to **log1p(jobs at the primary threshold only)**. `exceedance_prob_30min` / `_60min` compare the **same** `μ` draws to **thresholds on the 30- and 60-minute observed job distributions** (city-specific Q25 each). That is a **sensitivity on the desert cut**, not a full multi-outcome spatial model.
- **Optional fast MCMC:** set environment variable `PIPELINE_FAST_MCMC=1` for small `draws`/`tune` while debugging.
